In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == 'examples':
    package_dir = package_dir.parent
else:
    repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

import importlib
import ce_visualization_plotly
import ce_visualization_plotly.plugin as plotly_plugin

importlib.reload(plotly_plugin)
plotly_plugin.register_plotly_visualization_components()

# Plotly local factual bars

The default view shows signed contribution bars. Hover cards include the calibrated contribution interval and prediction metadata.

Key options:
- `uncertainty=True` (CE-native) or `show_uncertainty=True` — draw visible uncertainty bars
- `filter_top=N` — limit to the N highest-weight features
- `sort_by` — `"abs"` (default), `"value"`, `"interval_width"`, `"label"`, `"original"`
- `show_prediction_header=False` — hide the prediction band above the bars

This is a local factual plot, not a global explanation.

In [4]:
import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from calibrated_explanations import WrapCalibratedExplainer
import ce_visualization_plotly.plugin  # noqa: F401 - registers Plotly styles

## Classification

In [5]:
X_cls, y_cls = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=7,
)
x_proper, x_temp, y_proper, y_temp = train_test_split(
    X_cls,
    y_cls,
    test_size=0.40,
    random_state=7,
    stratify=y_cls,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_temp,
    y_temp,
    test_size=0.50,
    random_state=7,
    stratify=y_temp,
)

model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=7)
explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

factual = explainer.explain_factual(X_query[:3])

In [6]:
factual[0].plot(style="plotly.local.factual_bars", show=True);

In [7]:
# CE's uncertainty=True maps to show_uncertainty=True inside this plugin
factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    uncertainty=True,
)

PlotRenderResult(artifact={'artifact_type': 'plotly.local.factual_bars', 'artifact_version': '0.1.0', 'style': 'plotly.local.factual_bars', 'mode': 'classification', 'task': 'classification', 'prediction': {'value': 0.19047619047619047, 'low': 0.15, 'high': 0.2, 'label': '1.0', 'mode': 'classification', 'task': 'classification', 'kind': 'probabilistic', 'bars': [{'label': '1.0', 'value': 0.19047619047619047, 'low': 0.15, 'high': 0.2}, {'label': 'class 0', 'value': 0.8095238095238095, 'low': 0.8, 'high': 0.85}], 'x_range': [0.0, 1.0], 'x_label': 'Probability'}, 'items': [{'id': 'rule-0', 'rank': 0, 'feature_index': 1, 'feature_name': '1', 'rule': '1 <= -0.26', 'instance_value': np.float64(-0.8627926519554797), 'contribution': -0.5286935286935287, 'contribution_low': -0.5568137824235385, 'contribution_high': -0.5127758420441347, 'interval_width': 0.044037940379403784, 'crosses_zero': False, 'direction': 'negative', 'hover': 'Rule: 1 <= -0.26<br>Feature: 1<br>Feature index: 1<br>Current v

## Filtering and sorting

Use `filter_top` to limit to the N features with the largest contribution, and `sort_by` to control ordering.

Sorting by `"interval_width"` highlights the features whose contribution is most uncertain — useful when uncertainty bars are also enabled.

In [8]:
factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    filter_top=5,
    sort_by="interval_width",
    uncertainty=True,
);

## Hiding the prediction header

Set `show_prediction_header=False` to suppress the prediction band — useful when embedding the chart in a dashboard that already shows the prediction.

In [9]:
factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    show_prediction_header=False,
);

## Regression

In [10]:
X_reg, y_reg = make_regression(
    n_samples=600,
    n_features=8,
    n_informative=6,
    noise=8.0,
    random_state=11,
)
x_proper_reg, x_temp_reg, y_proper_reg, y_temp_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.40,
    random_state=11,
)
x_cal_reg, X_query_reg, y_cal_reg, y_query_reg = train_test_split(
    x_temp_reg,
    y_temp_reg,
    test_size=0.50,
    random_state=11,
)

reg_model = RandomForestRegressor(n_estimators=80, random_state=11)
reg_explainer = WrapCalibratedExplainer(reg_model)
reg_explainer.fit(x_proper_reg, y_proper_reg)
assert reg_explainer.fitted is True

reg_explainer.calibrate(x_cal_reg, y_cal_reg)
assert reg_explainer.calibrated is True

reg_factual = reg_explainer.explain_factual(
    X_query_reg[:3],
    low_high_percentiles=(10, 90),
)

In [11]:
reg_factual[0].plot(style="plotly.local.factual_bars", show=True)

PlotRenderResult(artifact={'artifact_type': 'plotly.local.factual_bars', 'artifact_version': '0.1.0', 'style': 'plotly.local.factual_bars', 'mode': 'regression', 'task': 'regression', 'prediction': {'value': -92.51986402735675, 'low': -138.86152376225223, 'high': -33.38992977973058, 'label': '1.0', 'mode': 'regression', 'task': 'regression', 'kind': 'regression', 'bars': [{'label': 'prediction', 'value': -92.51986402735675, 'low': -138.86152376225223, 'high': -33.38992977973058}], 'x_range': None, 'x_label': 'Predicted value'}, 'items': [{'id': 'rule-6', 'rank': 0, 'feature_index': 6, 'feature_name': '6', 'rule': '6 <= 0.16', 'instance_value': np.float64(-1.2818038176315936), 'contribution': -152.88346078899045, 'contribution_low': -212.0133950366166, 'contribution_high': -106.54180105409496, 'interval_width': 105.47159398252164, 'crosses_zero': False, 'direction': 'negative', 'hover': 'Rule: 6 <= 0.16<br>Feature: 6<br>Feature index: 6<br>Current value: -1.2818038176315936<br>Contribut

In [12]:
reg_factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    uncertainty=True,
    filter_top=6,
);

## HTML export

In [13]:
# Passing path= to .plot() can conflict with CE's internal kwarg handling.
# Use the returned PlotRenderResult to write the figure directly instead:
result = factual[0].plot(style="plotly.local.factual_bars", show=False)
result.figure.write_html("local_factual_bars.html")